# 01 - Data Exploration

**Purpose.** Load the joined Aave v3 / Compound v3 USDC hourly panel (`data/cached/joined_clean.parquet`) and visualise the raw inputs the forecaster will consume: rate series, cross-protocol spread, autocorrelation, TVL evolution, utilization distributions.

**Prerequisites.**
- `.venv` with `pandas`, `matplotlib`, `seaborn`, `statsmodels`.
- Optionally `data/cached/joined_clean.parquet`; otherwise a synthetic   panel is used (PROJECT_2_PLAN.md S4.1).

**Expected runtime.** < 30 seconds.


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Synthetic-data fallback: mirrors forecaster.train._make_synth_df.
# Used whenever the real joined_clean.parquet is not yet on disk.
import numpy as np
import pandas as pd


def make_synth_joined(n_rows: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_rows, freq="h", tz="UTC")

    def util_walk(start: float) -> np.ndarray:
        u = np.empty(n_rows)
        u[0] = start
        for i in range(1, n_rows):
            u[i] = np.clip(u[i - 1] + rng.normal(0.0, 0.01), 0.05, 0.97)
        return u

    u_a = util_walk(0.55)
    u_c = util_walk(0.45)

    # Toy rate process: kink-shaped baseline plus Gaussian residual.
    r_a = 0.05 * (u_a / 0.92) * 0.90 * u_a + rng.normal(0, 0.002, n_rows)
    r_c = 0.04 * u_c + rng.normal(0, 0.002, n_rows)
    r_a = np.clip(r_a, 0.0, 0.5)
    r_c = np.clip(r_c, 0.0, 0.5)

    tvl_a = 1e8 + np.cumsum(rng.normal(0, 1e5, n_rows))
    tvl_c = 5e7 + np.cumsum(rng.normal(0, 5e4, n_rows))
    gas = np.clip(
        20 + 5 * rng.standard_normal(n_rows) + 10 * np.sin(np.arange(n_rows) / 24),
        5, 200,
    )
    eth = 3000 + np.cumsum(rng.normal(0, 5, n_rows))

    return pd.DataFrame({
        "r_aave": r_a, "r_compound": r_c,
        "u_aave": u_a, "u_compound": u_c,
        "tvl_aave": tvl_a, "tvl_compound": tvl_c,
        "gas_gwei": gas, "eth_usd": eth,
    }, index=idx)


def load_joined(path: str = "data/cached/joined_clean.parquet") -> tuple[pd.DataFrame, bool]:
    """Try the real cached panel; fall back to synthetic on FileNotFoundError."""
    full = ROOT / path
    try:
        df = pd.read_parquet(full)
        print(f"[real] loaded {len(df):,} rows from {full}")
        return df, True
    except FileNotFoundError:
        print(f"[synth] {full} not found - generating synthetic panel")
        return make_synth_joined(), False


df, is_real = load_joined()
df.head()


In [ ]:
# Imports for plotting
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110


## 1. Rate series side-by-side

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(df.index, df['r_aave'] * 100, label='Aave v3', color='C0')
ax[1].plot(df.index, df['r_compound'] * 100, label='Compound v3', color='C1')
ax[0].set_ylabel('APY %')
ax[1].set_ylabel('APY %')
ax[1].set_xlabel('time (UTC)')
for a in ax:
    a.legend(loc='upper left')
fig.suptitle('USDC supply APY: Aave v3 vs Compound v3 (hourly, '
             f"{'real' if is_real else 'synthetic'} panel)")
fig.tight_layout()
plt.show()

# Caption: Both protocols' USDC supply rates over the 18-month window.
# Vertical alignment exposes co-movement; gaps in absolute level expose
# the cross-protocol dispersion the predictive allocator exploits.


## 2. Cross-protocol spread histogram

In [ ]:
spread = (df['r_aave'] - df['r_compound']) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(spread.dropna(), bins=60, edgecolor='black', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Aave APY - Compound APY (%)')
ax.set_ylabel('count of hourly bars')
ax.set_title('Hourly cross-protocol spread distribution')
plt.tight_layout()
plt.show()

print(f'Mean spread: {spread.mean():.4f}%')
print(f'Std  spread: {spread.std():.4f}%')
print(f'P(Aave > Compound): {(spread > 0).mean():.3f}')

# Caption: Distribution of hourly APY differences. The red dashed line
# at zero marks the no-edge point - mass on either side is potential
# capturable yield for an allocator that can predict the sign 12h ahead.


## 3. Autocorrelation of each rate (15 lags)

In [ ]:
try:
    from statsmodels.graphics.tsaplots import plot_acf
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    plot_acf(df['r_aave'].dropna(), lags=15, ax=ax[0], title='ACF: Aave')
    plot_acf(df['r_compound'].dropna(), lags=15, ax=ax[1], title='ACF: Compound')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('statsmodels not installed; skipping ACF plots')

# Caption: ACF up to 15h. Strong slow decay = persistent rate; AR(1)-
# like decay would justify naive last-observation baselines.


## 4. TVL evolution

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df['tvl_aave'] / 1e6, label='Aave TVL ($M)', color='C0')
ax.plot(df.index, df['tvl_compound'] / 1e6, label='Compound TVL ($M)', color='C1')
ax.set_ylabel('Total supplied USDC ($M)')
ax.set_xlabel('time (UTC)')
ax.legend()
ax.set_title('TVL evolution per protocol')
plt.tight_layout()
plt.show()

# Caption: TVL co-moves with broader USDC market interest, providing
# a regime variable separate from the rate process itself.


## 5. Utilization distributions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['u_aave'], bins=40, ax=ax[0], color='C0')
ax[0].set_title('Aave utilization')
ax[0].set_xlabel('u_aave')
sns.histplot(df['u_compound'], bins=40, ax=ax[1], color='C1')
ax[1].set_title('Compound utilization')
ax[1].set_xlabel('u_compound')
plt.tight_layout()
plt.show()

for proto in ('u_aave', 'u_compound'):
    print(f'{proto}: mean={df[proto].mean():.3f}  '
          f"q05={df[proto].quantile(0.05):.3f}  "
          f"q95={df[proto].quantile(0.95):.3f}")

# Caption: Utilization regimes - low utilization is rate-quiet,
# above-kink utilization is rate-volatile, and the MCDM f_Risk(u)
# factor penalises protocols pushed into the latter zone.


## Next steps

- Verify rate-spread sign and magnitude vs the PROJECT_2_PLAN.md S1.6   cointegration claim (Compound leads Aave).
- Proceed to `02_kink_calibration.ipynb` to extract per-protocol kink   parameters and verify residuals are bounded.

Relevant plan section: **PROJECT_2_PLAN.md S4 (Data Description)**.
